# SE446 — Quick Start: Run a Spark Job on the Cluster

This notebook lets you **submit Spark jobs to the SE446 YARN cluster** directly from Google Colab.

**How it works:** Colab connects to the master node via SSH, uploads your PySpark script, and runs `spark-submit` in client mode.

---

## Web UIs (open in browser)

| UI | URL |
|----|----- |
| YARN ResourceManager | https://hdfs.aniskoubaa.org/yarn/ |
| Spark History Server | https://hdfs.aniskoubaa.org/spark-history/ |
| Spark Web UI (live) | https://hdfs.aniskoubaa.org/spark-ui/ |
| HDFS NameNode | https://hdfs.aniskoubaa.org/ |

## How This Setup Works — Architecture

```
+------------------+         SSH          +-------------------+       YARN       +-------------------+
|                  | --------------------> |                   | ----------------> |                   |
|  Google Colab    |   (remote control)   |   Master Node     |   (schedules)    |   Worker Nodes    |
|                  |                      |                   |                  |                   |
|  - No Spark      |                      |  - spark-submit   |                  |  - Executor 1     |
|  - No Hadoop     |                      |  - DRIVER runs    |                  |  - Executor 2     |
|  - Just sends    |                      |    here (client   |                  |  - Process data   |
|    SSH commands   |                      |    mode)          |                  |    in parallel     |
+------------------+                      +-------------------+                  +-------------------+
```

### Key Concepts

- **Colab is NOT the Spark client.** It is only a remote terminal that sends commands via SSH to the master node.
- **Client mode** means the **Driver** process runs on the machine where `spark-submit` is executed — in our case, the **master node** (not Colab).
- **Executors** are launched by YARN on the **worker nodes** and do the actual data processing.
- `print()` output appears in Colab because SSH forwards the terminal output back from the master node.

### Why This Matters

| If you close...       | What happens?                                      |
|-----------------------|----------------------------------------------------|
| **Colab tab**         | SSH connection drops → Driver on master dies → **job fails** |
| **Nothing (keep open)** | Driver stays alive on master → job completes normally |

This is exactly the **client mode behavior** from the lecture: the Driver must stay connected. For long unattended jobs, use `--deploy-mode cluster` so the Driver runs inside YARN and survives disconnects.

## 1. Setup SSH Connection

**Two options** to connect to the cluster:

### Option A: Username & Password (recommended for students)
1. Click the **key icon** (🔑) in the left sidebar
2. Add a secret named `CLUSTER_PASSWORD` — your cluster password (provided by instructor)
3. Toggle **"Notebook access"** ON
4. Run the setup cell below — it will use your username and password

### Option B: SSH Key
1. Add a secret named `SSH_KEY` — paste the full contents of your private key file
2. Add a secret named `MASTER_HOST` — the cluster IP
3. Toggle **"Notebook access"** ON for both

In [ ]:
import os, getpass

# Install sshpass for password-based SSH
!apt-get -qq install -y sshpass > /dev/null 2>&1

MASTER_HOST = None
SSH_CMD = None

# --- Option A: Username + Password (via Colab Secrets or prompt) ---
try:
    from google.colab import userdata
    password = userdata.get('CLUSTER_PASSWORD')
    print("Password loaded from Colab Secrets.")
except Exception:
    password = None

if password is None:
    choice = input("No secret found. Enter 'p' for password login, 'k' for SSH key: ").strip().lower()
else:
    choice = 'p'

if choice == 'p':
    if password is None:
        password = getpass.getpass("Enter your cluster password: ")
    username = input("Enter your cluster username (e.g. akoubaa): ") if password else "akoubaa"
    try:
        MASTER_HOST = userdata.get('MASTER_HOST')
    except Exception:
        MASTER_HOST = input("Enter master node IP: ")
    SSH_CMD = f"sshpass -p '{password}' ssh -o StrictHostKeyChecking=no {username}@{MASTER_HOST}"
    SCP_CMD = f"sshpass -p '{password}' scp -o StrictHostKeyChecking=no"
    SCP_TARGET = f"{username}@{MASTER_HOST}"
    print(f"Configured: {username}@{MASTER_HOST} (password auth)")

# --- Option B: SSH Key ---
else:
    os.makedirs(os.path.expanduser("~/.ssh"), exist_ok=True)
    key_path = os.path.expanduser("~/.ssh/id_ed25519_hadoop_cluster")
    try:
        from google.colab import userdata
        ssh_key_content = userdata.get('SSH_KEY')
        MASTER_HOST = userdata.get('MASTER_HOST')
        with open(key_path, "w") as f:
            f.write(ssh_key_content)
            if not ssh_key_content.endswith("\n"):
                f.write("\n")
        os.chmod(key_path, 0o600)
        print("SSH key loaded from Colab Secrets.")
    except Exception:
        from google.colab import files
        print("Upload your SSH private key:")
        uploaded = files.upload()
        key_name = list(uploaded.keys())[0]
        with open(key_path, "wb") as f:
            f.write(uploaded[key_name])
        os.chmod(key_path, 0o600)
        MASTER_HOST = input("Enter master node IP: ")
    username = input("Enter username (default: root): ").strip() or "root"
    SSH_CMD = f"ssh -i {key_path} -o StrictHostKeyChecking=no {username}@{MASTER_HOST}"
    SCP_CMD = f"scp -i {key_path} -o StrictHostKeyChecking=no"
    SCP_TARGET = f"{username}@{MASTER_HOST}"
    print(f"Configured: {username}@{MASTER_HOST} (key auth)")

In [ ]:
# Cluster configuration (MASTER_HOST and SSH_KEY set in previous cell)
SSH_USER = "root"
SSH_CMD = f"ssh -i {SSH_KEY} -o StrictHostKeyChecking=no {SSH_USER}@{MASTER_HOST}"

# Test connection
!{SSH_CMD} 'echo "Connected to $(hostname)" && jps | grep -E "NameNode|ResourceManager|HistoryServer"'

## 2. Check Cluster Health

In [ ]:
# Check YARN nodes and HDFS status
!{SSH_CMD} 'export PATH=$PATH:/opt/hadoop/bin:/opt/spark/bin && \
  export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64 && \
  echo "=== YARN Nodes ===" && yarn node -list 2>/dev/null && \
  echo "" && echo "=== HDFS Data ===" && hdfs dfs -ls /data/ && \
  echo "" && echo "=== HDFS Summary ===" && hdfs dfs -df -h /'

## 3. Explore the Dataset

In [ ]:
# Preview the sample dataset (first 5 lines)
!{SSH_CMD} 'export PATH=$PATH:/opt/hadoop/bin && \
  export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64 && \
  echo "=== First 5 lines of chicago_crimes_sample.csv ===" && \
  hdfs dfs -cat /data/chicago_crimes_sample.csv | head -5 && \
  echo "" && echo "=== Line count ===" && \
  hdfs dfs -cat /data/chicago_crimes_sample.csv | wc -l'

## 4. Run a Simple Spark Job

This job reads the sample Chicago crimes dataset, counts crimes by type, and shows the top 10.

In [ ]:
%%writefile /tmp/crime_analysis.py
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, desc

spark = SparkSession.builder \
    .appName("SE446-QuickStart-CrimeAnalysis") \
    .getOrCreate()

# Read the sample dataset from HDFS
df = spark.read.csv("/data/chicago_crimes_sample.csv", header=True, inferSchema=True)

print(f"Total records: {df.count()}")
print(f"Number of columns: {len(df.columns)}")
print(f"Columns: {df.columns}")
print()

# Schema
df.printSchema()

# Top 10 crime types
print("=== Top 10 Crime Types ===")
df.groupBy("Primary Type") \
    .agg(count("*").alias("crime_count")) \
    .orderBy(desc("crime_count")) \
    .show(10, truncate=False)

# Arrest rate
total = df.count()
arrests = df.filter(col("Arrest") == "true").count()
print(f"Arrest rate: {arrests}/{total} = {arrests/total*100:.1f}%")

# Crimes by year
print("\n=== Crimes by Year ===")
df.groupBy("Year") \
    .agg(count("*").alias("crime_count")) \
    .orderBy("Year") \
    .show(20)

spark.stop()
print("\nDone!")

In [ ]:
# Upload script to master node
!{SCP_CMD} /tmp/crime_analysis.py {SCP_TARGET}:/tmp/crime_analysis.py
print("Script uploaded.")

In [ ]:
# Submit the Spark job to YARN in client mode
# While this runs, open the Spark Web UI: https://hdfs.aniskoubaa.org/spark-ui/
!{SSH_CMD} 'export PATH=$PATH:/opt/spark/bin:/opt/hadoop/bin && \
  export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64 && \
  export HADOOP_CONF_DIR=/opt/hadoop/etc/hadoop && \
  spark-submit \
    --master yarn \
    --deploy-mode client \
    --num-executors 2 \
    --executor-memory 768m \
    --executor-cores 1 \
    --driver-memory 512m \
    /tmp/crime_analysis.py'

## 5. Check Job in Spark History Server

After the job completes, open the **Spark History Server** to inspect the execution:

https://hdfs.aniskoubaa.org/spark-history/

Click on **SE446-QuickStart-CrimeAnalysis** to see:
- **Jobs tab** — how many jobs were triggered
- **Stages tab** — shuffle read/write, task distribution
- **SQL tab** — the query execution plan (DAG)

In [ ]:
# List recent YARN applications
!{SSH_CMD} 'export PATH=$PATH:/opt/hadoop/bin && \
  export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64 && \
  yarn application -list -appStates FINISHED 2>/dev/null | tail -10'

## 6. Try Your Own Script

Modify the cell below with your own PySpark code, then run the next cells to upload and submit it.

In [ ]:
%%writefile /tmp/my_spark_job.py
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, desc, avg

spark = SparkSession.builder \
    .appName("SE446-MyJob") \
    .getOrCreate()

df = spark.read.csv("/data/chicago_crimes_sample.csv", header=True, inferSchema=True)

# ---- Write your analysis here ----

# Example: Top 5 locations with most crimes
df.groupBy("Location Description") \
    .agg(count("*").alias("total")) \
    .orderBy(desc("total")) \
    .show(5, truncate=False)

# ---- End of your analysis ----

spark.stop()

In [ ]:
# Upload and submit your custom job
!{SCP_CMD} /tmp/my_spark_job.py {SCP_TARGET}:/tmp/my_spark_job.py

!{SSH_CMD} 'export PATH=$PATH:/opt/spark/bin:/opt/hadoop/bin && \
  export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64 && \
  export HADOOP_CONF_DIR=/opt/hadoop/etc/hadoop && \
  spark-submit \
    --master yarn \
    --deploy-mode client \
    --num-executors 2 \
    --executor-memory 768m \
    --executor-cores 1 \
    --driver-memory 512m \
    /tmp/my_spark_job.py'

---

## Reference Guide: Cluster Configuration & Web UIs

### Web Interfaces (all behind HTTPS with basic auth)

| Interface | URL | What It Shows |
|-----------|-----|---------------|
| **HDFS NameNode** | https://hdfs.aniskoubaa.org/ | File system browser, node health, storage usage |
| **YARN ResourceManager** | https://hdfs.aniskoubaa.org/yarn/ | Running/completed apps, cluster resources, node status |
| **Spark History Server** | https://hdfs.aniskoubaa.org/spark-history/ | Completed job details: stages, tasks, DAG, executor metrics |
| **Spark Web UI (live)** | https://hdfs.aniskoubaa.org/spark-ui/ | Live job info (only while a Spark job is running) |

### What to Look For in Each UI

- **YARN UI** — Click **Applications** to see submitted jobs, status, memory/CPU usage, and logs
- **Spark History** — Click a completed application to see the DAG, stages, shuffle read/write, and task distribution
- **Spark Web UI** — Only active during a running job. Shows live stages, storage (cached data), and SQL execution plan

### Available Data on HDFS

| File | Size | Description |
|------|------|-------------|
| `/data/chicago_crimes.csv` | ~174 MB | Full Chicago crimes dataset |
| `/data/chicago_crimes_sample.csv` | ~2.3 MB | Small sample (good for quick tests) |

**Columns:** `ID, Case Number, Date, Block, IUCR, Primary Type, Description, Location Description, Arrest, Domestic, Beat, District, Ward, Community Area, FBI Code, X Coordinate, Y Coordinate, Year, Updated On, Latitude, Longitude, Location`

### Cluster Specs

| Component | Value |
|-----------|-------|
| Spark version | 3.5.4 |
| Worker nodes | 2 (`worker-node-1`, `worker-node-2`) |
| Memory per worker (YARN) | 1536 MB |
| Default executor memory | 768 MB |
| Default executor instances | 2 |
| Default executor cores | 1 |
| Serializer | Kryo |

> **Important:** Do not request more than **768m per executor** or **1 core per executor** — the cluster is small. Requesting too much will cause YARN to queue your job indefinitely.

### Spark Configuration (`spark-defaults.conf`)

```
spark.master                     yarn
spark.submit.deployMode          client
spark.driver.memory              512m
spark.executor.memory            768m
spark.executor.instances         2
spark.executor.cores             1
spark.yarn.am.memory             256m
spark.eventLog.enabled           true
spark.eventLog.dir               hdfs:///spark-logs
spark.history.fs.logDirectory    hdfs:///spark-logs
spark.serializer                 org.apache.spark.serializer.KryoSerializer
spark.sql.warehouse.dir          hdfs:///user/hive/warehouse
spark.yarn.jars                  local:/opt/spark/jars/*
```

### Checking Results After a Job

1. **While running** — open https://hdfs.aniskoubaa.org/yarn/ to see the app under **Running**, and https://hdfs.aniskoubaa.org/spark-ui/ for live stages/tasks
2. **After completion** — open https://hdfs.aniskoubaa.org/spark-history/ and click the application to inspect stages, tasks, and performance
3. **YARN logs** (if using cluster mode): `yarn logs -applicationId <app-id>`